# 📊 Notebook 4 — Evaluation
**Crop Disease Diagnostician** | Accuracy Report

Generates: test accuracy, per-class F1, confusion matrix, F1 bar chart.

**Prerequisite**: Run notebooks 1 and 2 in this same session.
Needs: `/content/model/crop_disease_model.h5` and `/content/data/test/`

In [ ]:
!pip install -q seaborn scikit-learn

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import json, os

print(f'TF: {tf.__version__}')

MODEL_DIR  = '/content/model'
DATA_DIR   = '/content/data'
IMG_SIZE   = 224
BATCH_SIZE = 32

with open(f'{MODEL_DIR}/labels.json') as f:
    LABELS_MAP = json.load(f)
NUM_CLASSES = len(LABELS_MAP)
print(f'Classes: {NUM_CLASSES}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD MODEL
# ─────────────────────────────────────────────────────────────────────────────

MODEL_PATH = f'{MODEL_DIR}/crop_disease_model.h5'

if os.path.exists(MODEL_PATH):
    model = tf.keras.models.load_model(MODEL_PATH)
    print(f'✅ Loaded: {MODEL_PATH}')
else:
    print('Model not found. Run notebook 2 first or upload the .h5 file.')
    print('# Uncomment to upload:')
    print('# from google.colab import files; files.upload()')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD TEST DATASET FROM /content/data/test/
# (same fixed split created by notebook 1)
# ─────────────────────────────────────────────────────────────────────────────

ds_test = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/test',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,              # keep order for confusion matrix
    class_names=LABELS_MAP,    # enforce alphabetical class order
)

def preprocess_eval(image, label):
    image = tf.keras.applications.mobilenet_v2.preprocess_input(
        tf.cast(image, tf.float32)
    )
    return image, label

ds_test = (
    ds_test
    .map(preprocess_eval, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

# Count test samples
n_test = sum(1 for _ in ds_test.unbatch())
print(f'Test samples: {n_test}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN INFERENCE ON FULL TEST SET
# ─────────────────────────────────────────────────────────────────────────────

print('Running inference on test set...')
y_true, y_pred, y_conf = [], [], []

for images, labels in ds_test:
    preds        = model.predict(images, verbose=0)
    pred_classes = np.argmax(preds, axis=1)
    true_classes = np.argmax(labels.numpy(), axis=1) \
        if len(labels.shape) > 1 else labels.numpy()
    y_true.extend(true_classes.tolist())
    y_pred.extend(pred_classes.tolist())
    y_conf.extend(np.max(preds, axis=1).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

top1_acc  = accuracy_score(y_true, y_pred)
macro_f1  = f1_score(y_true, y_pred, average='macro')
mean_conf = y_conf.mean()

print(f'\n{"="*50}')
print(f'  TEST SET RESULTS — put these on your resume!')
print(f'{"="*50}')
print(f'  Top-1 Accuracy:  {top1_acc:.4f}  ({top1_acc*100:.2f}%)')
print(f'  Macro F1-Score:  {macro_f1:.4f}')
print(f'  Mean Confidence: {mean_conf:.4f}  ({mean_conf*100:.2f}%)')
print(f'  Test Samples:    {len(y_true)}')
print(f'{"="*50}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PER-CLASS CLASSIFICATION REPORT
# ─────────────────────────────────────────────────────────────────────────────

report = classification_report(y_true, y_pred, target_names=LABELS_MAP, digits=4)
print(report)

with open(f'{MODEL_DIR}/classification_report.txt', 'w') as f:
    f.write(f'Top-1 Accuracy: {top1_acc:.4f} ({top1_acc*100:.2f}%)\n')
    f.write(f'Macro F1:       {macro_f1:.4f}\n\n')
    f.write(report)
print('Saved: classification_report.txt')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFUSION MATRIX HEATMAP
# ─────────────────────────────────────────────────────────────────────────────

cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

short = ['Corn-GLS','Corn-Rust','Corn-H','Pot-EB','Pot-LB','Pot-H',
         'Tom-BS','Tom-EB','Tom-LB','Tom-LM','Tom-H']

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=short, yticklabels=short,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix (Acc: {top1_acc*100:.2f}%, F1: {macro_f1:.4f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# F1 BAR CHART
# ─────────────────────────────────────────────────────────────────────────────

per_class_f1 = f1_score(y_true, y_pred, average=None)
colors = ['#16a34a' if f>=0.95 else '#f59e0b' if f>=0.85 else '#ef4444'
          for f in per_class_f1]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(LABELS_MAP, per_class_f1, color=colors)
ax.set_xlim(0, 1.05)
ax.axvline(x=macro_f1, color='gray', ls='--', label=f'Macro avg: {macro_f1:.3f}')
ax.set_xlabel('F1-Score'); ax.set_title('Per-Class F1-Score', fontweight='bold')
ax.legend()
for bar, f in zip(bars, per_class_f1):
    ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
            f'{f:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/f1_per_class.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: f1_per_class.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SAVE METRICS JSON + DOWNLOAD
# ─────────────────────────────────────────────────────────────────────────────

import shutil
from google.colab import files

metrics = {
    'test_accuracy': round(float(top1_acc), 4),
    'macro_f1':      round(float(macro_f1), 4),
    'mean_confidence': round(float(mean_conf), 4),
    'test_samples':  int(len(y_true)),
    'num_classes':   NUM_CLASSES,
    'per_class_f1':  {k: round(float(v), 4) for k, v in zip(LABELS_MAP, per_class_f1)},
}
with open(f'{MODEL_DIR}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Summary:')
print(f'  Accuracy: {top1_acc*100:.2f}%')
print(f'  Macro F1: {macro_f1:.4f}')

shutil.make_archive('/content/evaluation_artifacts', 'zip', MODEL_DIR)
files.download('/content/evaluation_artifacts.zip')

print('\n✅ Notebook 4 complete! Check evaluation_artifacts.zip for all charts.')